In [1]:
import kagglehub, os

img_path  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
spl_path  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
code_path = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
ckpt_path = kagglehub.dataset_download("ashwanisingh997/run1-checkpoints-1229692")
for k, v in [("IMG", img_path), ("SPL", spl_path), ("CODE", code_path), ("CKPT", ckpt_path)]:
    print(f"{k}: {v}")
print("images:", len([f for f in os.listdir(img_path) if f.endswith('.jpg')]))

IMG: /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
SPL: /kaggle/input/datasets/ashwanisingh997/ham10000-splits-1229692
CODE: /kaggle/input/datasets/ashwanisingh997/thesis-code-1229692
CKPT: /kaggle/input/datasets/ashwanisingh997/run1-checkpoints-1229692
images: 10015


In [2]:
import hashlib

LAPTOP_DIGEST = "81280427151b71c7eb1cc79d78415265d6924ec30c27210ee0bd0c708672dbf2"
rows = []
for dirpath, _, files in os.walk(ckpt_path):
    for f in files:
        full = os.path.join(dirpath, f)
        h = hashlib.sha256()
        with open(full, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        rows.append((os.path.relpath(full, ckpt_path).replace(os.sep, "/"), h.hexdigest()))
rows.sort()
digest = hashlib.sha256("\n".join(f"{p}|{s}" for p, s in rows).encode()).hexdigest()
print(f"files: {len(rows)}  digest: {digest}")
assert len(rows) == 38 and digest == LAPTOP_DIGEST, "CHECKPOINTS DIFFER FROM THE INSPECTED SET"
print("CHECKPOINT MATCH")

files: 38  digest: 81280427151b71c7eb1cc79d78415265d6924ec30c27210ee0bd0c708672dbf2
CHECKPOINT MATCH


In [3]:
import importlib, importlib.metadata as md
try:
    print("before:", md.version("torchao"))
except md.PackageNotFoundError:
    print("before: not installed")

!pip uninstall -y -q torchao
importlib.invalidate_caches()
try:
    print("after:", md.version("torchao"))
except md.PackageNotFoundError:
    print("after: not installed")

import peft.import_utils as piu
if hasattr(piu.is_torchao_available, "cache_clear"):
    piu.is_torchao_available.cache_clear()
assert piu.is_torchao_available() is False, "torchao still visible to peft"
print("CELL 3 OK")

before: 0.10.0
after: not installed
CELL 3 OK


In [4]:
import sys, shutil, torch, yaml, pandas as pd

CODE, IMG, SPL, CKPT = code_path, img_path, spl_path, ckpt_path
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True)
os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    dst = f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}"
    shutil.copy(f"{CODE}/{f}", dst)
sys.path.insert(0, WORK)
os.chdir(WORK)

print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
for need in ["gan.py", "diffusion.py", "checkpointing.py", "data.py"]:
    assert need in os.listdir(f"{WORK}/src"), f"MISSING {need}"
tr = pd.read_csv(f"{SPL}/train.csv"); va = pd.read_csv(f"{SPL}/val.csv"); te = pd.read_csv(f"{SPL}/test.csv")
assert (len(tr), len(va), len(te)) == (6981, 1532, 1502), "splits differ from the frozen partition"

cfg = yaml.safe_load(open("config/config.yaml"))
from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="99ebf8d")

assert 'format={"jpg"' in open("src/checkpointing.py").read(), "OLD checkpointing.py - Kaggle code is not v3"
assert "pipe.vae.enable_slicing()" in open("src/diffusion.py").read(), "OLD diffusion.py - Kaggle code is not v3"
print("code version: v3 (99ebf8d) confirmed")
print("ALL CHECKS PASSED")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
code version: v3 (99ebf8d) confirmed
ALL CHECKS PASSED


In [5]:
import gc
from src.data import AUG_CLASSES
from src.gan import Generator
from src.diffusion import BASE, build_pipeline, generate_diffusion, generate_gan_images
from src.checkpointing import verify_synthetic_set

SMOKE = False   # True: 2 images per class as a test. False: the real 1,000 per class.
N   = 2 if SMOKE else cfg["synthesis"]["headline_per_class"]
OUT = "/kaggle/working/synthetic/" + ("smoke" if SMOKE else "run1_cloud")
LOG = f"{root}/logs"
CK  = f"{CKPT}/checkpoints"
print(f"N per class = {N}  ->  {OUT}")

# ---- Arm C: GAN, from gan_final.pt (D21) ----
G = Generator(latent_dim=cfg["gan"]["latent_dim"])
G.load_state_dict(torch.load(f"{CK}/gan_final.pt", map_location="cpu", weights_only=False)["models"]["G"])
generate_gan_images(G, AUG_CLASSES, N, f"{OUT}/gan_final",
                    latent_dim=cfg["gan"]["latent_dim"], base_seed=42,
                    manifest_path=f"{LOG}/sampling_gan_final{'_smoke' if SMOKE else ''}.json")
verify_synthetic_set(f"{OUT}/gan_final", AUG_CLASSES, N)
del G; gc.collect(); torch.cuda.empty_cache()

# ---- Arm D: LoRA diffusion, from lora_final (D21) ----
from diffusers import UNet2DConditionModel
from peft import PeftModel
unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16)
unet = PeftModel.from_pretrained(unet, f"{CK}/lora_final")
n_lora = sum(p.numel() for n, p in unet.named_parameters() if "lora_" in n)
print(f"[check] LoRA parameters loaded: {n_lora/1e6:.2f}M")
assert n_lora > 1e6, "adapter did not attach"
unet = unet.merge_and_unload().to(torch.float16)
pipe = build_pipeline(unet)

s = cfg["diffusion"]["sampler"]
generate_diffusion(pipe, AUG_CLASSES, N, f"{OUT}/lora_final", base_seed=42,
                   steps=s["num_inference_steps"], guidance=s["guidance_scale"],
                   size=cfg["data"]["image_size_diffusion"],
                   target_size=cfg["data"]["image_size_classifier"],
                   manifest_path=f"{LOG}/sampling_lora_final{'_smoke' if SMOKE else ''}.json",
                   session_limit_hours=10.5)
verify_synthetic_set(f"{OUT}/lora_final", AUG_CLASSES, N)
print("SAMPLING COMPLETE")

N per class = 1000  ->  /kaggle/working/synthetic/run1_cloud
[sample] akiec: 50 made, 0 skipped, 0.03s/image, ~0.04h left
[sample] akiec: 100 made, 0 skipped, 0.02s/image, ~0.02h left
[sample] akiec: 150 made, 0 skipped, 0.01s/image, ~0.02h left
[sample] akiec: 200 made, 0 skipped, 0.01s/image, ~0.02h left
[sample] akiec: 250 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 300 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 350 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 400 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 450 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 500 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 550 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 600 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 650 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 700 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 750 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] a

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

[check] LoRA parameters loaded: 1.59M


... (4 outputs left out)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[sample] akiec: 50 made, 0 skipped, 4.91s/image, ~6.75h left
[sample] akiec: 100 made, 0 skipped, 4.99s/image, ~6.79h left
[sample] akiec: 150 made, 0 skipped, 5.02s/image, ~6.76h left
[sample] akiec: 200 made, 0 skipped, 5.03s/image, ~6.71h left
[sample] akiec: 250 made, 0 skipped, 5.04s/image, ~6.65h left
[sample] akiec: 300 made, 0 skipped, 5.04s/image, ~6.59h left
[sample] akiec: 350 made, 0 skipped, 5.05s/image, ~6.52h left
[sample] akiec: 400 made, 0 skipped, 5.05s/image, ~6.45h left
[sample] akiec: 450 made, 0 skipped, 5.05s/image, ~6.39h left
[sample] akiec: 500 made, 0 skipped, 5.05s/image, ~6.32h left
[sample] akiec: 550 made, 0 skipped, 5.06s/image, ~6.25h left
[sample] akiec: 600 made, 0 skipped, 5.06s/image, ~6.18h left
[sample] akiec: 650 made, 0 skipped, 5.06s/image, ~6.11h left
[sample] akiec: 700 made, 0 skipped, 5.06s/image, ~6.04h left
[sample] akiec: 750 made, 0 skipped, 5.06s/image, ~5.98h left
[sample] akiec: 800 made, 0 skipped, 5.06s/image, ~5.91h left
[sample] 

In [6]:
import shutil
from PIL import Image

for arm in ["gan_final", "lora_final"]:
    for c in AUG_CLASSES:
        files = sorted(os.listdir(f"{OUT}/{arm}/{c}"))
        assert len([f for f in files if f.endswith(".png")]) == N, (arm, c, len(files))
        assert not any(f.endswith(".tmp") for f in files), f"leftover .tmp in {arm}/{c}"
        im = Image.open(f"{OUT}/{arm}/{c}/{files[0]}")
        assert im.size == (128, 128) and im.mode == "RGB", (arm, c, im.size, im.mode)
    print(f"{arm}: {N} x {len(AUG_CLASSES)} images, 128x128 RGB, no leftover .tmp")

shutil.make_archive("/kaggle/working/run1_synthetic_headline", "zip", "/kaggle/working", "synthetic")
print("saved: /kaggle/working/run1_synthetic_headline.zip")

gan_final: 1000 x 5 images, 128x128 RGB, no leftover .tmp
lora_final: 1000 x 5 images, 128x128 RGB, no leftover .tmp
saved: /kaggle/working/run1_synthetic_headline.zip
